# Модуль 8. Тестирование, рефакторинг, подготовка к защите

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр
**Модуль:** 8 — Тестирование, рефакторинг, подготовка к защите
**Теория:** 10 ч | **Практика:** 12 ч | **Самостоятельная работа:** 6 ч

---

## О чём этот модуль

Модуль 8 завершает курс. Код написан, паттерны внедрены, архитектура выстроена. Осталось три задачи:

1. **Убедиться, что всё работает** — тестирование.
2. **Сделать код чище** — рефакторинг.
3. **Уметь рассказать о своей работе** — подготовка к защите.

Этот модуль — не столько новые конструкции, сколько **дисциплина**: как довести проект до состояния, за которое не стыдно.

## Где мы находимся в проекте

После модуля 7 проект «Тайна исчезнувшего дива» имеет ECS, слои, async/threading и все шесть паттернов. Осталось отполировать.

| Что есть | Статус |
|---|---|
| Код и архитектура | Готовы (модули 1–7) |
| Юнит-тесты | Частично (по модулям) |
| Документация | Фрагментарна |
| Презентация | Нет |

## Что вы изучите

1. pytest: fixtures, parametrize, coverage.
2. Mocking: unittest.mock, monkeypatch.
3. TDD: Red-Green-Refactor.
4. Рефакторинг с сохранением поведения.
5. Code review: как принимать и давать замечания.
6. Документирование: README, ADR, UML.
7. Подготовка презентации и демонстрации.

## Содержание
1. Юнит-тестирование с pytest
2. Mocking
3. TDD
4. Рефакторинг с сохранением поведения
5. Code review
6. Документирование архитектуры
7. Подготовка к защите
8. Итоги и литература

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

for pkg in ["pytest", "pycodestyle"]:
    try:
        __import__(pkg)
        print(f"OK: {pkg}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# 1. Юнит-тестирование с pytest

## 1.1. Почему тесты важны

- Тесты — исполняемая документация: они показывают, как должен вести себя код.
- Тесты ловят регрессии: после рефакторинга сразу видно, что сломалось.
- Тесты дают смелость менять код.

## 1.2. fixtures

Фикстуры готовят повторяющиеся данные для тестов:

```python
import pytest

@pytest.fixture
def detective():
    return Detective("Аверин", health=100)

def test_detective_has_health(detective):
    assert detective.health == 100
```

## 1.3. parametrize

Один тест — много наборов данных:

```python
@pytest.mark.parametrize("hp,expected", [
    (100, True),
    (0, False),
    (-5, False),
])
def test_is_alive(hp, expected):
    assert Detective("x", hp).is_alive() is expected
```

## 1.4. Покрытие (coverage)

`pytest --cov=domain --cov-report=term` показывает, какой процент кода покрыт тестами. Цель — не 100%, а покрытие **ключевой логики**.

## Как это улучшит проект

Для «Тайны исчезнувшего дива» тестируем domain: позиции, здоровье, инвентарь, переходы состояний, фабрики. Не тестируем PyGame-отрисовку.

## Для углублённого изучения

- [Документация pytest](https://docs.pytest.org/).
- [pytest-cov](https://pypi.org/project/pytest-cov/).
- [Real Python: pytest](https://realpython.com/pytest-python-testing/).

---
# 2. Mocking

## 2.1. Зачем моки

Тесты должны быть быстрыми и детерминированными. Если `AudioManager.play` обращается к звуковой карте, тест будет медленным и хрупким. **Мок** заменяет внешнюю зависимость заглушкой.

## 2.2. unittest.mock

```python
from unittest.mock import Mock

audio = Mock()
audio.play("victory")

audio.play.assert_called_once_with("victory")
```

## 2.3. monkeypatch (pytest)

`monkeypatch` подменяет объекты на время теста:

```python
def test_load(monkeypatch):
    monkeypatch.setattr("pathlib.Path.read_bytes", lambda self: b"data")
    assert load_asset("a.png") == b"data"
```

## 2.4. Что мокать, а что нет

- **Мокать:** внешние системы — файлы, сеть, звук, время, рандом.
- **Не мокать:** domain-логику, которую и тестируем.

## Как это улучшит проект

`AudioManager` и загрузчик ассетов в тестах заменим моками — тесты станут быстрыми и независимыми от окружения.

## Для углублённого изучения

- Документация Python: [unittest.mock](https://docs.python.org/3/library/unittest.mock.html).
- Документация pytest: [monkeypatch](https://docs.pytest.org/en/stable/how-to/monkeypatch.html).

---
# 3. TDD: Red-Green-Refactor

## 3.1. Цикл

**TDD** (Test-Driven Development) — это разработка через тесты:

1. **Red** — пишем тест на ещё не существующую функцию. Тест падает.
2. **Green** — пишем минимальный код, чтобы тест прошёл.
3. **Refactor** — улучшаем код, тесты остаются зелёными.

## 3.2. Зачем

- Тест определяет **контракт** функции до её написания.
- Меньше «золотого кода» — пишем ровно то, что нужно.
- Каждая строка имеет тест.

## 3.3. Пример

```python
# Red: тест
def test_take_damage():
    detective = Detective("x", health=100)
    detective.take_damage(30)
    assert detective.health == 70

# Green: минимальная реализация
class Detective:
    def take_damage(self, amount):
        self.health -= amount
```

## Как это улучшит проект

Дополним тесты к игровой логике через TDD: сначала тест на поведение, потом реализация.

## Для углублённого изучения

- Beck K. «Test-Driven Development by Example».
- [Real Python: TDD](https://realpython.com/python-code-testing/).

---
# 4. Рефакторинг с сохранением поведения

## 4.1. Что такое рефакторинг

**Рефакторинг** — изменение внутренней структуры кода **без изменения поведения**. Если после рефакторинга что-то работает иначе — это уже не рефакторинг, а изменение.

## 4.2. Признаки плохого кода (smells)

- **Дублирование** — один и тот же код в двух местах.
- **Длинные методы** — метод на 200 строк.
- **Длинные списки параметров** — 7+ аргументов.
- **Божественный объект** — класс делает всё.
- **Магические числа** — `42` вместо именованной константы.

## 4.3. Безопасный рефакторинг

1. Убедитесь, что тесты проходят **до** рефакторинга.
2. Делайте **маленькие** шаги, коммитьте после каждого.
3. Прогоняйте тесты **после** каждого шага.
4. Не смешивайте рефакторинг и новые фичи.

## 4.4. Типичные приёмы

- **Extract Method** — вынести кусок в метод.
- **Rename** — понятное имя.
- **Replace Magic Number** — константа вместо числа.
- **Introduce Parameter Object** — группа параметров → объект.

## Как это улучшит проект

По итогам код-ревью преподавателя вы отрефакторите проблемные места, опираясь на тесты.

## Для углублённого изучения

- Fowler M. «Refactoring».
- [Refactoring Guru: smells](https://refactoring.guru/refactoring/smells).

---
# 5. Code review

## 5.1. Зачем

Код-ревью — это взгляд со стороны. Второй человек замечает то, что автор уже «замылил».

## 5.2. Как давать замечания

- Говорите о **коде**, не о человеке.
- Объясняйте **почему**, а не просто «плохо».
- Предлагайте **конкретное** решение.
- Отмечайте и хорошее.

## 5.3. Как принимать замечания

- Не воспринимайте замечания лично.
- Уточняйте, если непонятно.
- Спорьте аргументами, а не эмоциями.
- Исправляйте сразу или фиксируйте в TODO/ADR.

## 5.4. Чек-лист ревью

- Код читается сверху вниз?
- Имена отражают смысл?
- Нет дублирования?
- Зависимости идут в правильном направлении?
- Тесты покрывают изменения?

## Как это улучшит проект

На защите — код-ревью с преподавателем. Подготовьтесь: умейте объяснить каждое архитектурное решение.

## Для углублённого изучения

- [Google Engineering Practices: Code Review](https://google.github.io/eng-practices/review/).

---
# 6. Документирование архитектуры

## 6.1. README

Минимум в README итогового проекта:
- что за проект и как запустить;
- структура папок;
- какие паттерны и где;
- как запустить тесты.

## 6.2. ADR (Architecture Decision Records)

Журнал решений: ADR-001 ... ADR-007. Каждое — контекст, решение, альтернативы, последствия. Это показывает зрелость инженерной культуры.

## 6.3. UML и схемы

- Диаграмма классов — иерархия и связи.
- Диаграмма состояний — FSM игры.
- Схема слоёв — зависимости.
- Схема компонентов/систем — ECS.

## Как это улучшит проект

Документация — 5 баллов из 60 в итоговом проекте. Готовьте её постепенно, а не в последний день.

## Для углублённого изучения

- [ADR GitHub](https://adr.github.io/).
- [Mermaid](https://mermaid.js.org/) — диаграммы в Markdown.

---
# 7. Подготовка презентации и демонстрации

## 7.1. Структура выступления (5–7 минут)

1. **Интро (30 с):** тема, что за игра.
2. **Демонстрация (2–3 мин):** живой запуск, ключевые механики.
3. **Архитектура (2 мин):** слои, паттерны, ECS — что и почему.
4. **Сложные решения (1 мин):** где пришлось выбирать, что выбрали.
5. **Итоги (30 с):** что получилось, что бы улучшили.

## 7.2. Демонстрация

- Подготовьте **сценарий** демо: что показать за 3 минуты.
- Проверьте, что игра запускается **с чистого состояния**.
- Держите запасной вариант (скринкаст), если что-то упадёт.

## 7.3. Вопросы, к которым готовиться

- Почему выбрали этот паттерн, а не другой?
- Что изменилось бы при смене темы?
- Где границы слоёв и почему они там?
- Как тестировали? Что покрыто, что нет?

## Как это улучшит проект

Презентация + демонстрация + код-ревью — 15 баллов из 60. Это больше, чем документация.

## Для углублённого изучения

- [Правила хорошей презентации](https://www.nngroup.com/articles/presentations/).

---
# 8. Итоги курса

## Что вы прошли

| Модуль | Тема |
|---|---|
| 1–2 | ООП-основы, ABC, dataclasses, протоколы |
| 3 | SOLID |
| 4 | Game Loop, Update Method, Component |
| 5 | State, Strategy, Command |
| 6 | Observer, Factory, Singleton |
| 7 | ECS, MVC, слои, async/threading |
| 8 | Тестирование, рефакторинг, защита |

## Литература

**Базовая:**
- Документация pytest.
- Beck K. «Test-Driven Development by Example».

**Углублённая:**
- Fowler M. «Refactoring».
- [Refactoring Guru: smells](https://refactoring.guru/refactoring/smells).
- [Google Engineering Practices: Code Review](https://google.github.io/eng-practices/review/).

## Что дальше

В практической работе вы финализируете проект: тесты, документация, презентация — и готовитесь к защите.